In [1]:
import os 
os.chdir("../")

In [2]:
from langchain.document_loaders import PyPDFLoader,DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

c:\Users\Gurleen\.conda\envs\medbot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Extract text from PDF files
def load_pdf_files(data):
    loader =DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    documents = loader.load()
    return documents


In [4]:
extracted_data = load_pdf_files("data")


In [5]:
#extracted_data

In [6]:
#to filter out the actual data from the pdf files
 
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    to filter out - given a list of documnent objects, return a new list of document objects 
    that contain only the source in metadata and original page content.
    """
    minimal_docs: List[Document] = []
    for doc in docs:
        src= doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content =doc.page_content,
                metadata={"Source":src}
            )
        )
    return minimal_docs

In [7]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [8]:
# minimal_docs

In [9]:
# split the documents into smaller chunks
#Chunking 
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
        length_function=len
    )
    texts_chunk= text_splitter.split_documents(minimal_docs)
    return texts_chunk


In [10]:
texts_chunk= text_split(minimal_docs)
print(f"Number of chunks:{len(texts_chunk)}")


Number of chunks:5859


In [11]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    model_name= "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name= model_name
    )
    return embeddings

embedding = download_embeddings()


C:\Users\Gurleen\AppData\Local\Temp\ipykernel_27560\3883129348.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [12]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [13]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
# OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
# os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

In [14]:
from pinecone import Pinecone
pinecone_api_key = os.getenv("PINECONE_API_KEY")
pc= Pinecone(api_key=pinecone_api_key)

In [15]:
from pinecone import ServerlessSpec

index_name= "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec= ServerlessSpec(cloud="aws",region="us-east-1")
    )

index= pc.Index(index_name)


In [16]:
from langchain_pinecone import PineconeVectorStore

stats = index.describe_index_stats()

# upload only if index is empty 
if stats.total_vector_count == 0:
    print("Index is empty, uploading documents...")

    docsearch= PineconeVectorStore.from_documents(
        documents=texts_chunk,
        embedding=embedding,
        index_name=index_name
    )
else:
    print("Doc exists, skipping upload")
    docsearch= PineconeVectorStore(
        index_name=index_name,
        embedding=embedding
        )

Doc exists, skipping upload


In [17]:
# Load Existing index

from langchain_pinecone import PineconeVectorStore
#Embed each chunk and upsert the mebeddings into your pinecone index

docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
    )

In [18]:
#Add more data to existing Pinecone index

# dswith= Document(
#     page_content="The non-communicable disease is a disease which is not caused by germs and not transmitted from one to another. This is caused by some improper functioning of the body organs",
#     metadata={"Source":"Google"}
# )

In [19]:
# docsearch.add_documents(documents=[dswith])

In [20]:
retriever = docsearch.as_retriever(search_type="similarity",search_kwargs={"k":5})


In [21]:
retrived_docs= retriever.invoke("what is paracetamol medicine used for?")
print(retrived_docs)

[Document(id='45debd27-2ac2-4697-bfc7-808c140e6af4', metadata={'Source': 'data\\Medical_book.pdf'}, page_content='tion. 330 C Street SW, Washington, DC 20447. (800) 392-\n3366.\nOTHER\nElder Abuse Prevention. <http://www.oaktrees.org/elder>.\nNational Institute on Drug Abuse. <http://www.nida.nih.gov>.\nLaith Farid Gulli, M.D.\nBilal Nasser, M.Sc.\nAcceleration-deceleration cervical injury\nsee Whiplash\nACE inhibitors see Angiotensin-converting\nenzyme inhibitors\nAcetaminophen\nDefinition\nAcetaminophen is a medicine used to relieve pain\nand reduce fever.\nPurpose\nAcetaminophen is used to relieve many kinds of'), Document(id='257c7a65-cfdb-4231-9686-cb0c5f7f2279', metadata={'Source': 'data\\Medical_book.pdf'}, page_content='terol-lowering drug cholestyramine (Questran), the\nantibiotic Isoniazid, and zidovudine (Retrovir, AZT).\nGALE ENCYCLOPEDIA OF MEDICINE 2 19\nAcetaminophen\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 19'), Document(id='1b255717-e8af-45f1-9961-319bc2ef9477',

In [27]:
# from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama import ChatOllama

chatmodel= ChatOllama(model= "llama3.2:1b",
                      temperature=0,
        )

In [28]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate


In [29]:
system_prompt=(
    "You are a Medical Assistant for question-answering tasks."
    "Use the following pieces of retrieved context to answer"
    "all the question. If you don't know the answer , say that you don't know."
    "Use four sentences maximum and keep the answer concise."
    "\n\n"
    "{context}"

    # "You are a medical assistant."
    # "Answer the user's question using ONLY the provided context."
    # "If the question asks about multiple topics, answer ALL of them."
    # "Provide:"
    # "1. Definition"
    # "2. Causes"
    # "3. Symptoms"
    # "4. Treatment (if available)"
     
)
prompt = ChatPromptTemplate.from_messages(
    [
    ("system",system_prompt),
    ("human","{input}"),
    ]
)


In [30]:
# creating a  rag chain
question_answer_chain= create_stuff_documents_chain(chatmodel, prompt)
rag_chain=  create_retrieval_chain(retriever, question_answer_chain)


In [31]:
response= rag_chain.invoke({"input":"what are Acromegaly and gigantism?"})
print(response["answer"])

Acromegaly and gigantism are two related but distinct medical conditions.

**Acromegaly:**
Acromegaly is a disorder in which the pituitary gland releases excess growth hormone (GH) into the bloodstream. This excess GH causes abnormal growth of bones, soft tissues, and organs, leading to various symptoms such as:

* Enlarged hands and feet
* Joint pain and swelling
* Skin thickening and wrinkles
* Vision problems
* Sleep apnea

Acromegaly typically occurs in people who have a normal or low level of GH production, but the pituitary gland overproduces GH.

**Gigantism:**
Gigantism is a rare condition that occurs when the body produces too much growth hormone (GH) after bone growth has stopped. This can happen at any age, but it's more common in children and adolescents. Gigantism causes:

* Enlarged bones
* Excessive hair growth
* Increased appetite
* Sleep apnea

Gigantism is often caused by a tumor on the pituitary gland or an overproduction of GH.

**Key differences:**

* Acromegaly oc

In [32]:
response=rag_chain.invoke({"input":"what is paracetamol medicine used for?"})
print(response["answer"])

Acetaminophen, commonly known as paracetamol, is a medicine used to relieve pain and reduce fever. It is used to treat various types of headaches, muscle aches, and minor pains such as toothaches or menstrual cramps.


In [ ]:
# response= rag_chain.invoke({"input":"what is Balantidiasis and also explain use of Aspirin?"})
# print(response["answer"])

In [ ]:
# response= rag_chain.invoke({"input":"what is the treatment for diabetes?"})
# print(response["answer"])